In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
spotify_green = '#1DB954'
spotify_black = '#191414'
spotify_dark_grey = '#212121'
spotify_white = '#FFFFFF'

In [ ]:
tracks = pd.read_csv('spotify-tracks-dataset.csv')
user = pd.read_csv('spotify_user_behavior_realistic_50000_rows.csv')
top50 = pd.read_csv('spotify-streaming-top-50-world.csv')
spain = pd.read_csv('spotify-streaming-top-50-spain.csv')

In [ ]:
dataframes = {
    "Tracks Dataset": tracks,
    "User Behavior": user,
    "Top 50 World": top50,
    "Top 50 Spain": spain
}

for name, df in dataframes.items():
    print(f"--- {name} Columns ---")
    print(list(df.columns))
    print("\n" + "="*30 + "\n")

In [ ]:
tracks.info()
user.info()
top50.info()
spain.info()

In [ ]:
print(f"Συνολικά υπάρχουν {len(tracks)} τραγούδια στο dataset.")

rows, columns = tracks.shape
print(f"Το dataset έχει {rows} γραμμές και {columns} στήλες.")

## Καθαρισμός δεδομένων (Data cleaning)

In [ ]:
tracks.isnull().sum()

In [ ]:
tracks = tracks.dropna()

print(tracks.isnull().sum())

In [ ]:
user.isnull().sum()

In [ ]:
top50.isnull().sum()

In [ ]:
spain.isnull().sum()

In [ ]:
duplicates_count = tracks.duplicated(subset=['track_id']).sum()
print(f"Βρέθηκαν {duplicates_count} διπλότυπα tracks.")

tracks = tracks.drop_duplicates(subset=['track_id'], keep='first')


In [ ]:
tracks = tracks.loc[:, ~tracks.columns.str.contains('^Unnamed')]

In [ ]:
print(tracks[['duration_ms', 'popularity', 'danceability']].describe())

tracks = tracks[tracks['duration_ms'] > 10000]

In [ ]:
tracks = tracks.reset_index(drop=True)

In [ ]:
user = user.drop_duplicates(subset=['user_id'])

In [ ]:
print(user['age'].describe())

In [ ]:
top50= top50.sort_values('date', ascending=False).drop_duplicates('song')

## Οπτικοποίηση δεδομένων

In [ ]:
fig = px.scatter(
    tracks, 
    x='track_genre', 
    y='danceability',
    color='track_genre',           
    hover_name='track_name',       
    title='Ανάλυση Danceability ανά Μουσικό Είδος',
    labels={
        'track_genre': 'Είδος Μουσικής', 
        'danceability': 'Επίπεδο Danceability'
    },
    template='plotly_white',
    opacity=0.7                    
)

In [ ]:
fig.update_layout(
    paper_bgcolor=spotify_black,
    
    plot_bgcolor=spotify_dark_grey,
    
    
    title_font=dict(color=spotify_white, size=24),
    font=dict(color=spotify_white),


    xaxis=dict(
        tickangle=-45,                
        gridcolor='#333333',           
        linecolor=spotify_white,         
        title_font=dict(color=spotify_white)
    ),
    yaxis=dict(
        gridcolor='#333333',
        linecolor=spotify_white,
        title_font=dict(color=spotify_white),
        range=[0, 1]                  
    ),
    
    showlegend=False
)

In [ ]:
pio.renderers.default = "browser"
fig.show()

# Plot (Θηκόγραμμα)

Το **Box Plot** συμπυκνώνει χιλιάδες σημεία δεδομένων σε ένα απλό σχήμα που δείχνει την κατανομή τους.

### 1. Τα 5 Σημεία-Κλειδιά
Κάθε "κουτί" στο γράφημα αντιπροσωπεύει 5 βασικά στατιστικά στοιχεία:

* **Minimum (Ελάχιστο):** Η χαμηλότερη τιμή (εξαιρουμένων των outliers).
* **Πρώτο Τεταρτημόριο (Q1 - 25%):** Το 25% των τραγουδιών έχουν danceability κάτω από αυτό το σημείο.
* **Διάμεσος (Median - 50%):** Η γραμμή στη μέση του κουτιού. Χωρίζει τα δεδομένα ακριβώς στη μέση.
* **Τρίτο Τεταρτημόριο (Q3 - 75%):** Το 75% των τραγουδιών έχουν danceability κάτω από αυτό το σημείο.
* **Maximum (Μέγιστο):** Η υψηλότερη τιμή (εξαιρουμένων των outliers).

### 2. Τι σημαίνει το σχήμα;
* **Το μήκος του κουτιού (IQR):** Όσο πιο πλατύ είναι το κουτί, τόσο μεγαλύτερη ποικιλία (διασπορά) έχουν τα τραγούδια αυτού του είδους.
* **Η θέση της Διαμέσου:** Αν η γραμμή είναι προς τα δεξιά (κοντά στο 1.0), τότε το είδος είναι κατά βάση πολύ χορευτικό.

### 3. Outliers (Ακραίες Τιμές)
Οι μεμονωμένες τελείες που εμφανίζονται πέρα από τις "μουστάκες" (whiskers) ονομάζονται **outliers**. Είναι τραγούδια που αποτελούν εξαίρεση στον κανόνα του είδους τους (π.χ. ένα πολύ "danceable" τραγούδι σε ένα είδος που συνήθως είναι αργό).

---

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_genres = tracks['track_genre'].value_counts().nlargest(15).index
filtered_tracks = tracks[tracks['track_genre'].isin(top_genres)]

plt.figure(figsize=(12, 8), facecolor='#191414') 
ax = plt.gca()
ax.set_facecolor('#212121') 

sns.boxplot(
    data=filtered_tracks, 
    x='danceability', 
    y='track_genre', 
    palette='viridis', 
    order=filtered_tracks.groupby('track_genre')['danceability'].median().sort_values().index
)

plt.title('Κατανομή Danceability ανά Είδος', color='white', fontsize=16, pad=20)
plt.xlabel('Danceability', color='white', fontsize=12)
plt.ylabel('Είδος Μουσικής', color='white', fontsize=12)

plt.xticks(color='white')
plt.yticks(color='white')

for spine in ax.spines.values():
    spine.set_visible(False)

plt.grid(axis='x', color='#333333', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.io as pio

top_genres = tracks['track_genre'].value_counts().nlargest(15).index
filtered_tracks = tracks[tracks['track_genre'].isin(top_genres)]

fig = px.box(
    filtered_tracks, 
    x='danceability', 
    y='track_genre', 
    color='track_genre',
    title='Κατανομή Danceability στα 15 Δημοφιλέστερα Είδη',
    labels={'track_genre': 'Είδος', 'danceability': 'Danceability'},
    template='plotly_dark'
)

fig.update_layout(
    showlegend=False,
    paper_bgcolor='#191414',
    plot_bgcolor='#212121',
    yaxis={'categoryorder':'total ascending'} 
)

fig.show()

In [ ]:

top10_genres = tracks['track_genre'].value_counts().nlargest(10).index
filtered_tracks = tracks[tracks['track_genre'].isin(top10_genres)]


fig = px.scatter_3d(
    filtered_tracks, 
    x='danceability', 
    y='energy', 
    z='popularity',
    color='track_genre',           
    hover_name='track_name',      
    opacity=0.5,                   
    title='Διαδραστική 3D Ανάλυση: Danceability, Energy & Popularity'
)

fig.update_layout(
    template='plotly_dark',         
    paper_bgcolor='#191414',
    scene = dict(
        xaxis=dict(title='Danceability', backgroundcolor='#212121'),
        yaxis=dict(title='Energy', backgroundcolor='#212121'),
        zaxis=dict(title='Popularity', backgroundcolor='#212121')
    ),
    legend=dict(title='Είδος Μουσικής', yanchor="top", y=0.99, xanchor="left", x=0.01)
)

pio.renderers.default = "notebook" 
fig.show()

# 3D Scatter - Radar Chart (Αράχνη)

### Overplotting
Στο προηγούμενο 3D γράφημα, η προσπάθεια να απεικονίσουμε χιλιάδες τραγούδια ταυτόχρονα οδήγησε σε ένα δυσνόητο αποτέλεσμα. Λόγω της μεγάλης στατιστικής επικάλυψης στις τιμές danceability και energy, τα σημεία στριμώχνονται.

### Radar Chart 
Αντί να κοιτάμε κάθε τραγούδι ξεχωριστά, χρησιμοποιούμε τους **μέσους όρους** των χαρακτηριστικών ανά μουσικό είδος. Το Radar Chart μας επιτρέπει να δούμε το μοναδικό "αποτύπωμα" κάθε είδους σε έναν κυκλικό άξονα.



#### Πώς διαβάζεται το γράφημα:
* **Κορυφές (Axes):** Κάθε γωνία αντιπροσωπεύει μια μεταβλητή (`Danceability`, `Energy`, `Popularity`).
* **Εμβαδόν (Area):** Όσο πιο μεγάλη είναι η επιφάνεια που καλύπτει ένα είδος, τόσο υψηλότερες τιμές έχει συνολικά στα χαρακτηριστικά αυτά.
* **Σύγκριση:** Εκεί που τα σχήματα τέμνονται ή αποκλίνουν, φαίνονται οι πραγματικές διαφορές (π.χ. αν η *Pop* έχει περισσότερο Energy από την *Acoustic*).

### 3. Προετοιμασία Δεδομένων
Για να είναι η σύγκριση δίκαιη, η `Popularity` (κλίμακα 0-100) κανονικοποιείται στην κλίμακα **0-1**, ώστε να είναι συμβατή με το `Danceability` και το `Energy`.

In [ ]:

top5_genres = tracks['track_genre'].value_counts().nlargest(5).index
df_radar = tracks[tracks['track_genre'].isin(top5_genres)]
df_radar = df_radar.groupby('track_genre')[['danceability', 'energy', 'popularity']].mean().reset_index()


df_radar['popularity'] = df_radar['popularity'] / 100

fig = go.Figure()

for i, row in df_radar.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['danceability'], row['energy'], row['popularity'], row['danceability']],
        theta=['Danceability', 'Energy', 'Popularity', 'Danceability'],
        fill='toself',
        name=row['track_genre']
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1], gridcolor='#333333'),
        bgcolor='#212121'
    ),
    paper_bgcolor='#191414',
    font=dict(color='white'),
    showlegend=True,
    title='Σύγκριση Μέσων Όρων ανά Είδος'
)

fig.show()

## Top 50 Bar chart

In [ ]:
import plotly.graph_objects as go 

top50sorted = top50.sort_values('position').head(50)

trace = go.Bar(
    x=top50sorted['song'],
    y=60 - top50sorted['position'],
    text=top50sorted['position'],
    marker=dict(color='#1DB954'),
    name = 'World top 50 songs'
)
layout = go.Layout(
    title='World top 50 songs',
    xaxis=dict(title='Songs', tickangle=-45), 
    yaxis=dict(title='Position'),
    template='plotly_dark',
    paper_bgcolor='#191414',
    plot_bgcolor='#212121',
    margin=dict(b=150)
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

## Παρατηρούμε πολλά 4άρια 5άρια κλπ γιατί εμφανίζει τα κορυφαία τραγούδια από πολλές διαφορετικές χώρες.
* Για να βελτιώσω το διάγραμμα θα ταξινομήσω με βάση την πιό πρόσφατη ημερομηνία

In [ ]:
latest_date = top50['date'].max()

top50sorted = top50[top50['date'] == latest_date].sort_values('position').head(50)

trace = go.Bar(
    x=top50sorted['song'],
    y=60 - top50sorted['position'],
    text=top50sorted['position'],
    marker=dict(color='#1DB954'),
    name = 'World top 50 songs'
)
layout = go.Layout(
    title='World top 50 songs',
    xaxis=dict(title='Songs', tickangle=-45), 
    yaxis=dict(title='Position'),
    template='plotly_dark',
    paper_bgcolor='#191414',
    plot_bgcolor='#212121',
    margin=dict(b=150)
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

## KMeans Clustering: Mood Groups
* Elbow Method -> find the optimal number of clusters.

In [ ]:
FEATURE_COLS = ['danceability', 'energy', 'loudness', 'speechiness','instrumentalness', 'liveness', 'valence', 'acousticness', 'tempo']

scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(tracks[FEATURE_COLS])

#elbow method
inertias    = []
sil_scores  = []
K_range     = range(2, 11)

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score']
)
fig.add_trace(go.Scatter(
    x=list(K_range), y=inertias,
    mode='lines+markers',
    line=dict(color='#1DB954', width=3),
    marker=dict(size=9, color='#1DB954')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(K_range), y=sil_scores,
    mode='lines+markers',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=9, color='#FF6B6B')
), row=1, col=2)

fig.update_layout(
    title=dict(text='📐 Finding Optimal Number of Clusters', font_size=18),
    height=420,
    paper_bgcolor='#191414',
    plot_bgcolor='#191414',
    font=dict(color='#FFFFFF', family='Arial'),
    showlegend=False
)
for axis in ['xaxis','xaxis2','yaxis','yaxis2']:
    fig.layout[axis].update(gridcolor='#333', color='#FFFFFF')

fig.show()
print(f'Best Silhouette Score: {max(sil_scores):.3f} at K={list(K_range)[sil_scores.index(max(sil_scores))]}')

Συμφωνα με τα παραπάνω διαγράμματα θα επιλέξουμε 4 συστάδες για να κατηγοριοποιήσουμε τα δεδομένα
1. ⚡ High-Energy Party (Dance & Pop)
Αυτή η κατηγορία αφορά τη δυναμική, ρυθμική μουσική που ακούγεται σε clubs ή ραδιόφωνα.
* Υψηλό danceability, energy, loudness και tempo.

* Valence: Υψηλό (θετική, ανεβαστική διάθεση).

* Acousticness: Πολύ χαμηλό.

2. 🌙 Chill & Melancholic (Acoustic / Ballads)
Εδώ ανήκουν τα τραγούδια που βασίζονται στην ηρεμία και το συναίσθημα, συνήθως με φυσικά όργανα.

* Πολύ υψηλό acousticness.

* Energy / Loudness: Χαμηλά (πιο "σιγανά" κομμάτια).

* Valence: Χαμηλό προς μέτριο (μελαγχολική ή ήρεμη διάθεση).

3. 🌑 Dark & Aggressive (Rock / Metal / Techno)
Μουσική με μεγάλη ένταση που όμως δεν είναι "χαρούμενη". Είναι η "σκοτεινή" πλευρά της ενέργειας.

* Μέγιστο energy και loudness.

* Valence: Πολύ χαμηλό (θυμός, ένταση, σκοτάδι).

* Instrumentalness: Συχνά υψηλό (αν είναι techno/industrial) ή χαμηλό (αν είναι metal).

4. 🧠 Deep Focus / Instrumental (Ambient / Study Beats)
Η κατηγορία για συγκέντρωση, όπου ο στίχος δεν πρέπει να αποσπά την προσοχή.

*  Μέγιστο instrumentalness και ελάχιστο speechiness.

* Liveness: Χαμηλό (συνήθως studio παραγωγές).

* Energy: Σταθερό, μέτριο προς χαμηλό.

In [ ]:
k = 4

kmeans        = KMeans(n_clusters=k, random_state=42, n_init=10)
tracks['cluster'] = kmeans.fit_predict(X_scaled)

cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURE_COLS)
print('Cluster Centers:')
print(cluster_centers.round(3))


MOOD_LABELS = {
    0: ' Party Anthems',
    1: ' Dark & Intense',
    2: ' Chill & Acoustic',
    3: ' Deep Focus / Instrumental'
}

centers_tracks = cluster_centers.copy()
centers_tracks['cluster_id'] = range(k)
centers_tracks = centers_tracks.sort_values('energy', ascending=False).reset_index(drop=True)

auto_labels = [
    '⚡ High-Energy Party',      
    '🌑 Dark & Intense',         
    '🌙 Chill & Acoustic',       
    '🧠 Deep Focus / Instrumental' 
]
cluster_name_map = dict(zip(centers_tracks['cluster_id'].values, auto_labels))
tracks['mood'] = tracks['cluster'].map(cluster_name_map)

print(f'\nSongs per mood cluster:')
print(tracks['mood'].value_counts())

In [ ]:
pca        = PCA(n_components=2, random_state=42)
X_pca      = pca.fit_transform(X_scaled)
tracks['pca1'] = X_pca[:, 0]
tracks['pca2'] = X_pca[:, 1]

print(f'Explained variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.1f}%')

fig = px.scatter(
    tracks,
    x='pca1', y='pca2',
    color='mood',
    hover_name='track_name',
    hover_data = [
    'artists',          # Ποιος το τραγουδάει
    'album_name',       # Σε ποιο άλμπουμ ανήκει
    'track_genre',      # Το είδος 
    'popularity',       # Πόσο γνωστό είναι
    'energy',           # Τεχνικό χαρακτηριστικό
    'valence',          # Η θετικότητα του τραγουδιού
    'tempo'             # Η ταχύτητα 
],
    color_discrete_sequence=px.colors.qualitative.Vivid,
    title=' KMeans Clusters — 2D PCA View of All Songs',
    labels={'pca1': 'PCA Component 1', 'pca2': 'PCA Component 2', 'mood': 'Mood Cluster'}
)
fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.update_layout(
    height=560,
    paper_bgcolor=spotify_black,
    plot_bgcolor='#1a1a2e',
    font=dict(color=spotify_white, family='Arial'),
    title_font_size=18,
    legend=dict(bgcolor='rgba(0,0,0,0)', title_text='Mood Cluster'),
    xaxis=dict(gridcolor='#2a2a2a', color=spotify_white),
    yaxis=dict(gridcolor='#2a2a2a', color=spotify_white)
)
fig.show()

## Content-Based Recommendation Model 
* Το μοντέλο χρησιμοποιεί τη Συνημίτονη Ομοιότητα (Cosine Similarity) για να συγκρίνει τα κανονικοποιημένα χαρακτηριστικά πχ. ενέργεια, ρυθμό των τραγουδιών.
* Μετρά τη γωνία μεταξύ των διανυσμάτων τους στον μαθηματικό χώρο: όσο *μικρότερη η γωνία*, τόσο πιο *παρόμοια είναι η αύρα των κομματιών*, επιτρέποντας στο σύστημα να προτείνει τραγούδια με παρεμφερή ήχο.

In [ ]:
def recommend_songs(track_name_query, n=5, same_mood=True):
    """
    Προτείνει τραγούδια παρόμοια με το track_name_query βάσει του similarity_matrix.
    """
    matches = tracks[tracks['track_name'].str.lower().str.contains(track_name_query.lower())]
    
    if matches.empty:
        print(f'❌ Το τραγούδι "{track_name_query}" δεν βρέθηκε.')
        print('Δοκίμασε κάποιο από αυτά:', tracks['track_name'].head(5).tolist())
        return None

    song_idx = matches.index[0]
    song_info = tracks.loc[song_idx]
    
    print(f'🎵 Input Song: {song_info["track_name"]} — {song_info["artists"]}')
    print(f'   Genre: {song_info["track_genre"]} | Mood: {song_info["mood_label"]}')
    print(f'   Energy: {song_info["energy"]} | Valence: {song_info["valence"]} | Tempo: {song_info["tempo"]}')
    print('-' * 70)

    raw_scores = cosine_similarity(X_scaled[song_idx].reshape(1, -1), X_scaled).flatten()
    
    sim_scores = list(enumerate(raw_scores))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = [s for s in sim_scores if s[0] != song_idx]
    if same_mood:
        mood_scores = [s for s in sim_scores 
                       if tracks.loc[s[0], 'mood_label'] == song_info['mood_label']]

        if len(mood_scores) >= n:
            sim_scores = mood_scores

    top_n = sim_scores[:n]

    results = []
    for idx, score in top_n:
        row = tracks.loc[idx]
        results.append({
            'Rank': len(results) + 1,
            'Song': row['track_name'],
            'Artist': row['artists'],
            'Genre': row['track_genre'],
            'Mood': row['mood_label'],
            'Similarity': round(score, 4),
            'Popularity': row['popularity'],
            'Energy': row['energy'],
            'Valence': row['valence']
        })

    return pd.DataFrame(results)

print('✅ Η συνάρτηση recommend_songs είναι έτοιμη για το dataset σου!')

✅ Η συνάρτηση recommend_songs είναι έτοιμη για το dataset σου!


In [69]:
MOOD_MAPPING = {
    0: '🧠 Deep Focus / Instrumental',
    1: '⚡ High-Energy Party',
    2: '🌙 Chill & Acoustic',
    3: '🌑 Dark & Intense'
}

tracks['mood_label'] = tracks['cluster'].map(MOOD_MAPPING)

print("Στήλες που υπάρχουν τώρα:", tracks.columns.tolist())
print("\nΔείγμα δεδομένων:")
print(tracks[['track_name', 'mood_label']].head())

Στήλες που υπάρχουν τώρα: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'cluster', 'mood', 'pca1', 'pca2', 'mood_label']

Δείγμα δεδομένων:
                   track_name           mood_label
0                      Comedy  ⚡ High-Energy Party
1            Ghost - Acoustic   🌙 Chill & Acoustic
2              To Begin Again   🌙 Chill & Acoustic
3  Can't Help Falling In Love   🌙 Chill & Acoustic
4                     Hold On   🌙 Chill & Acoustic


In [70]:
recs_weeknd = recommend_songs('Blinding Lights', n=5)
recs_weeknd

🎵 Input Song: Blinding Lights — Kidz Bop Kids
   Genre: children | Mood: ⚡ High-Energy Party
   Energy: 0.775 | Valence: 0.867 | Tempo: 171.011
----------------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood,Similarity,Popularity,Energy,Valence
0,1,魅せてくれ - Remastered 2022,Eikichi Yazawa,rockabilly,⚡ High-Energy Party,0.9996,23,0.691,0.756
1,2,Mob Rule,Bad//Dreems,garage,⚡ High-Energy Party,0.9992,29,0.794,0.892
2,3,PØBEL,Crashville,country,⚡ High-Energy Party,0.9992,42,0.742,0.809
3,4,Júpiter,Los Vaguens,rockabilly,⚡ High-Energy Party,0.9990,22,0.663,0.722
4,5,Se Fiquei Esperando Meu Amor Passar,Legião Urbana,mpb,⚡ High-Energy Party,0.9990,43,0.576,0.680


In [71]:
weeknd_songs = tracks[tracks['artists'].str.contains('The Weeknd', case=False, na=False)]

print(f"Βρέθηκαν {len(weeknd_songs)} εγγραφές για τον The Weeknd.")
weeknd_songs[['track_name', 'popularity', 'mood_label', 'track_genre']].sort_values('popularity', ascending=False).head(20)

Βρέθηκαν 48 εγγραφές για τον The Weeknd.


,track_name,popularity,mood_label,track_genre
67616,Blinding Lights,91,⚡ High-Energy Party,pop
67617,Starboy,90,⚡ High-Energy Party,pop
67654,I Was Never There,90,⚡ High-Energy Party,pop
67696,Save Your Tears,89,⚡ High-Energy Party,pop
67682,Call Out My Name,89,⚡ High-Energy Party,pop
67857,Die For You,88,⚡ High-Energy Party,pop
67845,The Hills,88,⚡ High-Energy Party,pop
15975,Lost in the Fire (feat. The Weeknd),85,⚡ High-Energy Party,club
19002,You Right,84,⚡ High-Energy Party,dance
19301,Moth To A Flame (with The Weeknd),83,⚡ High-Energy Party,dance


In [72]:
recs_weeknd = recommend_songs('I Was Never There', n=5)
recs_weeknd

🎵 Input Song: woo x I was never there (tiktok version) — untrusted;creamy;11:11 Music Group
   Genre: chill | Mood: 🌙 Chill & Acoustic
   Energy: 0.58 | Valence: 0.282 | Tempo: 112.05
----------------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood,Similarity,Popularity,Energy,Valence
0,1,Dizzy On the Comedown,Turnover,emo,🌙 Chill & Acoustic,0.9993,60,0.632,0.265
1,2,He Lives in You,Lebo M.,disney,🌙 Chill & Acoustic,0.9988,23,0.632,0.290
2,3,He Lives in You,Lebo M.,disney,🌙 Chill & Acoustic,0.9988,43,0.632,0.290
3,4,He Lives in You,Lebo M.,disney,🌙 Chill & Acoustic,0.9988,18,0.632,0.290
4,5,Christmas Here With You,Four Tops;Aretha Franklin,disco,🌙 Chill & Acoustic,0.9981,0,0.579,0.255
